# Recent Feature Additions

## Version 0.1.12

### DuckDB-powered simulation engine

A new high-performance simulation engine backed by DuckDB enables ERAD to scale to **million+ assets**:

- **Vectorized computation**: All hazard vector calculations (earthquake, wind, flood, fire) are expressed as SQL CROSS JOINs with custom UDFs, replacing the previous Python loop-based approach.
- **Custom UDFs**: Haversine distance, Holland wind model, MMI/PGA/PGV calculations, and CDF evaluation are registered as native DuckDB functions for SIMD-accelerated processing.
- **Automatic parallelism**: DuckDB uses all available CPU cores for computation without requiring explicit multiprocessing.
- **Multiple export formats**: Results can be exported directly to Parquet (ZSTD compressed), SQLite, CSV, PyArrow Table, or pandas DataFrame.
- **Direct array loading**: For million+ asset scale, assets can be loaded from numpy arrays or DataFrames without Pydantic object overhead.
- **API preservation**: The existing `HazardSimulator.run()` interface is unchanged — the engine selection is controlled by the `engine` parameter (`'duckdb'` or `'legacy'`).

### GDM interface improvements

- Updated `AssetSystem.from_gdm()` to support the latest GDM component types.
- Improved coordinate extraction and elevation handling during model loading.

### Dependency updates

- Added `duckdb>=1.0` and `pyarrow` as core dependencies.
- Pinned `grid-data-models==2.3.7`.

In [ ]:
from erad.engine import SimulationEngine

# Initialize the DuckDB-powered engine
engine = SimulationEngine()
print(f"Engine initialized: {type(engine).__name__}")

### Using the DuckDB engine via HazardSimulator

The DuckDB engine is the new default. Use `engine='legacy'` to fall back to the original loop-based approach.

In [ ]:
from erad.runner import HazardSimulator

# DuckDB engine (default)
# sim = HazardSimulator(asset_system, engine='duckdb')
# sim.run(hazard_system)

# Legacy engine (for comparison/debugging)
# sim = HazardSimulator(asset_system, engine='legacy')
# sim.run(hazard_system)

print("Available engine modes: 'duckdb' (default), 'legacy'")

### Scale workloads (100K+ assets)

For large-scale simulations, skip Pydantic object hydration and use the engine's export methods directly:

In [ ]:
import numpy as np
from uuid import uuid4
from datetime import datetime
from shapely.geometry import Point
from infrasys.quantities import Distance
from erad.engine import SimulationEngine
from erad.systems.hazard_system import HazardSystem
from erad.models.hazard.earthquake import EarthQuakeModel
from erad.default_fragility_curves import DEFAULT_FRAGILTY_CURVES

# Generate 10K synthetic assets
N = 10_000
np.random.seed(42)
asset_ids = [str(uuid4()) for _ in range(N)]
asset_names = [f'pole_{i}' for i in range(N)]
asset_types = np.full(N, 6, dtype=np.int32)  # distribution_poles
asset_type_names = ['distribution_poles'] * N
lats = 36.0 + np.random.random(N)
lons = -121.0 + np.random.random(N)
heights = np.full(N, 10.0)
elevations = np.zeros(N)

# Create hazard
eq = EarthQuakeModel(
    name='eq1',
    timestamp=datetime(2024, 1, 1),
    origin=Point(-120.5, 36.5),
    depth=Distance(15, 'km'),
    magnitude=5.5,
)
hazard_system = HazardSystem(name='demo')
hazard_system.add_component(eq)

# Run via engine directly (fastest path)
engine = SimulationEngine()
engine.load_assets_from_arrays(
    asset_ids, asset_names, asset_types, asset_type_names,
    lats, lons, heights, elevations
)
engine.load_hazards(hazard_system)
engine.load_fragility_curves(DEFAULT_FRAGILTY_CURVES)
engine._loaded = True
engine.run()

# Export results
df = engine.export_to_dataframe()
print(f"Computed {len(df):,} asset-state records")
print(f"Mean survival probability: {df['survival_probability'].mean():.4f}")
print(f"Min survival probability: {df['survival_probability'].min():.6f}")
df.head()

## Version 0.1.11

### MCP server

Introduction of a Model Context Protocol (MCP) server for ERAD, enabling AI agents and language models to interact with simulation capabilities through standardized tools:

- **`erad-mcp`**: CLI entry point exposing ERAD functionality as MCP tools.
- **Core tools**: `load_distribution_model`, `run_simulation`, `generate_scenarios`, `export_to_json`, `export_to_sqlite`, and more.
- **Query tools**: `list_asset_types`, `query_assets`, `get_asset_details`, `get_asset_statistics`, `get_network_topology`.
- **Hazard tools**: `list_historic_earthquakes`, `list_historic_hurricanes`, `list_historic_wildfires`, `load_historic_earthquake`, `load_historic_hurricane`, `load_historic_wildfire`.
- **Documentation search**: `search_documentation` tool for querying ERAD docs.

### CLI interface

New command-line interface via `typer`:

- `erad run` — Run hazard simulations from the command line.
- `erad export` — Export results to various formats.
- `erad list-hazards` — List available historic hazard events.

### Dependency updates

- Added `mcp>=1.0.0`, `typer>=0.9.0`, `rich>=13.0.0`.

In [ ]:
# MCP server can be started via CLI:
# $ erad-mcp

# Or used programmatically:
from erad.mcp import TOOLS
print(f"MCP server exposes {len(TOOLS)} tools")

## Version 0.1.10

### Multi-hazard simulation

ERAD now supports simultaneous multi-hazard scenarios:

- Earthquake, wind (hurricane), flood, and wildfire hazards can be combined in a single simulation.
- Survival probability is computed as the product of independent per-hazard survival probabilities.
- `HazardScenarioGenerator` produces Monte Carlo damage scenarios accounting for all active hazards.

### Hydrological hazard parameters

Extended flood model with additional hydrological parameters:

- `soil_saturation` — Ratio of soil moisture content.
- `snow_water_equivalent` — Depth of water if all snow were melted.
- `runoff_volume` — Surface runoff flow rate.
- `groundwater_flow` — Subsurface groundwater flow rate.

### Default fragility curves

Comprehensive set of default fragility curves for all 14 asset types across 6 hazard parameters:

- Wind speed, flood depth, flood velocity, fire boundary distance, PGA, and PGV.
- Based on published literature (EPRI, PNNL, IEEE Transactions on Power Systems).
- Users can override with custom `HazardFragilityCurves` definitions.

In [ ]:
from erad.default_fragility_curves import DEFAULT_FRAGILTY_CURVES

print(f"Default fragility curve sets: {len(DEFAULT_FRAGILTY_CURVES)}")
for fc in DEFAULT_FRAGILTY_CURVES:
    print(f"  - {fc.asset_state_param}: {len(fc.curves)} asset type curves")